In [3]:
pip install bert-score transformers torch

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
from bert_score import BERTScorer

In [ ]:
reference_dir = "/Users/senudaliyanage/Downloads/IIT/Final Year Project/Legal Document Summarization/results/openai_results/openai_summaries_test_merged"
model_dir = "/Users/senudaliyanage/Downloads/IIT/Final Year Project/Legal Document Summarization/results/mistral24B_results/mistral24B_summaries_test_merged"

In [6]:
references = []
predictions = []
documents = []
skipped = []

for file in sorted(os.listdir(reference_dir)):
    if not file.endswith(".txt"):
        continue

    ref_path = os.path.join(reference_dir, file)
    pred_path = os.path.join(model_dir, file)

    if not os.path.exists(pred_path):
        continue

    with open(ref_path, encoding="utf-8") as f:
        ref = f.read().strip()
    with open(pred_path, encoding="utf-8") as f:
        pred = f.read().strip()

    if not ref or not pred:
        skipped.append(file)
        continue

    references.append(ref)
    predictions.append(pred)
    documents.append(file)

print(f"Loaded  : {len(references)} document pairs")
print(f"Skipped : {len(skipped)} empty files")

Loaded  : 373 document pairs
Skipped : 0 empty files


In [7]:
from transformers import AutoTokenizer, AutoModel

model_name = "microsoft/deberta-base-mnli"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.model_max_length = 512

print("Downloading model...")
model = AutoModel.from_pretrained(model_name)

print("Model cached and ready")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 3228.25it/s, Materializing param=encoder.rel_embeddings.weight]                     
DebertaModel LOAD REPORT from: microsoft/deberta-base-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
classifier.bias     | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 
config              | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model cached and ready


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/transfor

In [ ]:
from bert_score import BERTScorer

scorer = BERTScorer(
    model_type="microsoft/deberta-base-mnli",
    device="cpu",
    batch_size=4
)

scorer._tokenizer.model_max_length = 512

P, R, F1 = scorer.score(predictions, references, verbose=True)

print(f"\nBERTScore Results (Mistral 24B vs OpenAI reference)")
print(f"Precision : {P.mean():.4f}")
print(f"Recall    : {R.mean():.4f}")
print(f"F1        : {F1.mean():.4f}")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 3253.06it/s, Materializing param=encoder.rel_embeddings.weight]                     
DebertaModel LOAD REPORT from: microsoft/deberta-base-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
classifier.bias     | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 
config              | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/12 [00:00<?, ?it/s]Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/Users/senudaliyanage/miniconda3/envs/IRP/lib/python3.10/site-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
  File "/Users/senudaliyanage/miniconda3/envs/IRP

computing greedy matching.


100%|██████████| 6/6 [00:00<00:00, 14.40it/s]

done in 125.53 seconds, 2.97 sentences/sec

BERTScore Results (Mistral 32B vs OpenAI reference)
Precision : 0.7164
Recall    : 0.6962
F1        : 0.7053
